In [21]:
import pandas as pd

train = pd.read_csv("~/assignment1/assignment-1/data/train.csv")
test = pd.read_csv("~/assignment1/assignment-1/data/test.csv")

print("Training shape:", train.shape)
print("Test shape:", test.shape)

print("\nTraining columns:")
print(train.columns.tolist())

print("\nTraining examples:")
print(train.head(5).to_string())

print("\nMissing values:")
print(train.isnull().sum())

Training shape: (10, 7)
Test shape: (4, 7)

Training columns:
['File Name', 'Aggrement Value', 'Aggrement Start Date', 'Aggrement End Date', 'Renewal Notice (Days)', 'Party One', 'Party Two']

Training examples:
                                                    File Name  Aggrement Value Aggrement Start Date Aggrement End Date  Renewal Notice (Days)                                               Party One                 Party Two
0  6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1             6500           20.05.2007         20.05.2008                   15.0  Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles     GERALDINE Q. GALINATO
1         6683129-House-Rental-Contract-Geraldine-Galinato-v2             6500           20.05.2007         20.05.2008                   15.0  Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles     GERALDINE Q. GALINATO
2                                 18325926-Rental-Agreement-1             4000           05.12.2008         31.11.20

In [22]:

from pathlib import Path


BASE_DIR = Path.home() / "assignment1" / "assignment-1"
DATA_DIR = BASE_DIR / "data"

print("Assignment folder:", BASE_DIR)
print("Data folder exists:", DATA_DIR.exists())

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTraining files:")
for p in sorted((DATA_DIR / "train").iterdir()):
    print(p.name)

print("\nTest files:")
for p in sorted((DATA_DIR / "test").iterdir()):
    print(p.name)


Assignment folder: /home/sahil/assignment1/assignment-1
Data folder exists: True
Train shape: (10, 7)
Test shape: (4, 7)

Training files:
18325926-Rental-Agreement-1.docx
36199312-Rental-Agreement.png
44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx
46239065-Standard-Rental-Agreement-Rental-With-Performance-Fee.docx
47854715-RENTAL-AGREEMENT.docx
50070534-RENTAL-AGREEMENT.docx
54770958-Rental-Agreement.png
54945838-Rental-Agreement.png
6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx
6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx

Test files:
156155545-Rental-Agreement-Kns-Home.pdf.docx
228094620-Rental-Agreement.pdf.docx
24158401-Rental-Agreement.png
95980236-Rental-Agreement.png


In [23]:


def find_document(name, folder):
    name = str(name).strip().casefold()

    for path in Path(folder).iterdir():
        if not path.is_file():
            continue

        filename = path.name.casefold()

        
        if filename.endswith(".pdf.docx"):
            base_name = filename[:-len(".pdf.docx")]
        else:
            base_name = path.stem.casefold()

        if base_name == name:
            return path

    return None


print("TRAIN")
for name in train["File Name"]:
    path = find_document(name, DATA_DIR / "train")

    if path is None:
        path = find_document(name, DATA_DIR / "test")

    print(name, "->", path)

print("\nTEST")
for name in test["File Name"]:
    print(name, "->", find_document(name, DATA_DIR / "test"))



TRAIN
6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1 -> /home/sahil/assignment1/assignment-1/data/train/6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx
6683129-House-Rental-Contract-Geraldine-Galinato-v2 -> /home/sahil/assignment1/assignment-1/data/train/6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx
18325926-Rental-Agreement-1 -> /home/sahil/assignment1/assignment-1/data/train/18325926-Rental-Agreement-1.docx
24158401-Rental-Agreement -> /home/sahil/assignment1/assignment-1/data/test/24158401-Rental-Agreement.png
36199312-Rental-Agreement -> /home/sahil/assignment1/assignment-1/data/train/36199312-Rental-Agreement.png
44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement -> /home/sahil/assignment1/assignment-1/data/train/44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx
47854715-RENTAL-AGREEMENT -> /home/sahil/assignment1/assignment-1/data/train/47854715-RENTAL-AGREEMENT.docx
50070534-RENTAL-AGREEMENT -> /home/sahil/assignment1/assignment-1/d

In [24]:

%pip install pandas python-docx pillow pytesseract


Note: you may need to restart the kernel to use updated packages.


In [25]:

from docx import Document
from PIL import Image
import pytesseract


BASE_DIR = Path.home() / "assignment1" / "assignment-1"
DATA_DIR = BASE_DIR / "data"

def read_document(path):
    path = Path(path)
    ext = path.suffix.lower()

    if ext == ".docx":
        doc = Document(path)
        parts = []

        for p in doc.paragraphs:
            if p.text.strip():
                parts.append(p.text.strip())

        for table in doc.tables:
            for row in table.rows:
                parts.append(
                    " | ".join(cell.text.strip() for cell in row.cells)
                )

        return "\n".join(parts)

    if ext in {".png", ".jpg", ".jpeg"}:
        return pytesseract.image_to_string(
            Image.open(path).convert("RGB")
        )

    raise ValueError(f"Unsupported file format: {path.name}")



sample_path = (
    DATA_DIR / "train" / "18325926-Rental-Agreement-1.docx"
)

sample_text = read_document(sample_path)

print("Extracted characters:", len(sample_text))
print(sample_text[:3000])


Extracted characters: 2952
RENTAL AGREEMENT
This deed of rental agreement executed at Bangalore this fifth day of December 2008 between MR.K.Kuttan S/o Kelu Aehari (Late) residing at site No 152 Geethalayam OMH colong S.M. Road 1st main, T.Dasarahalli, Bangalore-57. Here after called the lessor which term shall where the contest so admits mean and includes his legal heirs representatives executors and assign and Sri. P.M. Narayana Namboodri “Laxmi Leela” ground floor 3rd cross Ayyappa Nagar behind Ayyappa Temple, Jalahalli West, Bangalore - 15 referred as tenant which term shall wherever the context so admits mean and includes his legal heirs representations executors assign witnesses.
Whereas the lessor is the absolute owner after premises at site No. 8/20 S.M. Road, Jalahalli West, Bangalore - 15 and he has constructed approximate 500 sft area of Ilnd floor with A.C. Sheet roof with power light in the above said premises.
Where as the owner has offered to let out the premises for non

In [ ]:


def locate_file(name, folder):
    name = str(name).strip().casefold()

    for path in Path(folder).iterdir():
        if not path.is_file():
            continue

        filename = path.name.casefold()

        if filename.endswith(".pdf.docx"):
            base = filename[:-len(".pdf.docx")]
        else:
            base = path.stem.casefold()

        if base == name:
            return path

    return None


train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

records = []

for split, df in [("train", train), ("test", test)]:
    for _, row in df.iterrows():
        path = locate_file(row["File Name"], DATA_DIR / split)

        if path is None and split == "train":
            
            path = locate_file(row["File Name"], DATA_DIR / "test")

        if path is None:
            print("FILE NOT FOUND:", row["File Name"])
            continue

        try:
            text = read_document(path)
            records.append({
                "split": split,
                "file_name": row["File Name"],
                "path": str(path),
                "text": text,
                "characters": len(text)
            })
            print("OK:", path.name, "characters:", len(text))

        except Exception as e:
            print("ERROR:", path.name, str(e))

text_df = pd.DataFrame(records)

output_dir = BASE_DIR / "outputs"
output_dir.mkdir(exist_ok=True)

text_df.to_csv(
    output_dir / "extracted_text.csv",
    index=False
)

print("\nTotal extracted:", len(text_df))
print(text_df[["split", "file_name", "characters"]])


OK: 6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx characters: 3192
OK: 6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx characters: 16865
OK: 18325926-Rental-Agreement-1.docx characters: 2952
OK: 24158401-Rental-Agreement.png characters: 3406
OK: 36199312-Rental-Agreement.png characters: 2298
OK: 44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx characters: 1080
OK: 47854715-RENTAL-AGREEMENT.docx characters: 4558
OK: 50070534-RENTAL-AGREEMENT.docx characters: 2677
OK: 54770958-Rental-Agreement.png characters: 3421
OK: 54945838-Rental-Agreement.png characters: 3485
OK: 24158401-Rental-Agreement.png characters: 3406
OK: 95980236-Rental-Agreement.png characters: 3386
OK: 156155545-Rental-Agreement-Kns-Home.pdf.docx characters: 4194
OK: 228094620-Rental-Agreement.pdf.docx characters: 5070

Total extracted: 14
    split                                          file_name  characters
0   train  6683127-House-Rental-Contract-GERALDINE-GALINA...        3192
1   

In [ ]:



BASE_DIR = Path.home() / "assignment1" / "assignment-1"
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
text_df = pd.read_csv(OUTPUT_DIR / "extracted_text.csv")

print("Train rows:", len(train))
print("Test rows:", len(test))

duplicates = set(train["File Name"]) & set(test["File Name"])
print("Files appearing in both sets:", duplicates)

train_data = train.merge(
    text_df[text_df["split"] == "train"][["file_name", "text"]],
    left_on="File Name",
    right_on="file_name",
    how="left"
)

test_data = test.merge(
    text_df[text_df["split"] == "test"][["file_name", "text"]],
    left_on="File Name",
    right_on="file_name",
    how="left"
)

print("\nMissing training text:", train_data["text"].isna().sum())
print("Missing test text:", test_data["text"].isna().sum())

print("\nTraining labels:")
print(train_data[[
    "File Name",
    "Aggrement Value",
    "Aggrement Start Date",
    "Aggrement End Date",
    "Renewal Notice (Days)",
    "Party One",
    "Party Two"
]].to_string(index=False))


Train rows: 10
Test rows: 4
Files appearing in both sets: {'24158401-Rental-Agreement'}

Missing training text: 0
Missing test text: 0

Training labels:
                                                 File Name  Aggrement Value Aggrement Start Date Aggrement End Date  Renewal Notice (Days)                                              Party One                             Party Two
6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1             6500           20.05.2007         20.05.2008                   15.0 Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles                 GERALDINE Q. GALINATO
       6683129-House-Rental-Contract-Geraldine-Galinato-v2             6500           20.05.2007         20.05.2008                   15.0 Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles                 GERALDINE Q. GALINATO
                               18325926-Rental-Agreement-1             4000           05.12.2008         31.11.2009                   90.0             